# Vietnamese Medical VQA
## Short-answer evaluation notebook

Notebook này dùng trực tiếp dataset Hugging Face `SpringWang08/medical-vqa-vi`,
chuẩn hóa đầu ra theo thuật ngữ y khoa và kiểm soát câu trả lời tối đa 10 từ.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import yaml
from datasets import load_dataset

from src.utils.text_utils import count_words, get_target_answer, postprocess_answer

PROJECT_ROOT = Path(".").resolve()
CONFIG_PATH = PROJECT_ROOT / "configs" / "medical_vqa.yaml"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

ANSWER_MAX_WORDS = int(cfg["data"].get("answer_max_words", 10))
REPO_ID = cfg["data"]["hf_dataset"]
dataset = load_dataset(REPO_ID)

print("Loaded dataset:", REPO_ID)
for split_name, split_ds in dataset.items():
    print(f"- {split_name}: {len(split_ds)} samples")
print("answer_max_words =", ANSWER_MAX_WORDS)

In [ ]:
def summarize_split(split_ds):
    summary = {
        "size": len(split_ds),
        "sources": {},
        "answer_types": {},
        "modalities": {},
        "target_lengths": [],
    }
    for item in split_ds:
        source = item.get("source", "unknown")
        answer_type = item.get("answer_type", "unknown")
        modality = item.get("modality", "") or "unknown"
        target = get_target_answer(item, max_words=ANSWER_MAX_WORDS)

        summary["sources"][source] = summary["sources"].get(source, 0) + 1
        summary["answer_types"][answer_type] = summary["answer_types"].get(answer_type, 0) + 1
        summary["modalities"][modality] = summary["modalities"].get(modality, 0) + 1
        summary["target_lengths"].append(count_words(target))
    return summary

split_summaries = {name: summarize_split(ds) for name, ds in dataset.items()}
split_summaries

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

train_summary = split_summaries["train"]

axes[0].bar(train_summary["sources"].keys(), train_summary["sources"].values(), color="#4C78A8")
axes[0].set_title("Train split by source")
axes[0].set_ylabel("Samples")

axes[1].bar(train_summary["answer_types"].keys(), train_summary["answer_types"].values(), color="#F58518")
axes[1].set_title("Train split by answer type")

modality_items = sorted(train_summary["modalities"].items(), key=lambda item: item[1], reverse=True)[:6]
axes[2].bar([k for k, _ in modality_items], [v for _, v in modality_items], color="#54A24B")
axes[2].set_title("Top modalities in train split")
axes[2].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(split_summaries["train"]["target_lengths"], bins=range(1, ANSWER_MAX_WORDS + 3), color="#E45756", edgecolor="black")
ax.axvline(ANSWER_MAX_WORDS, color="black", linestyle="--", label=f"max_words={ANSWER_MAX_WORDS}")
ax.set_title("Distribution of normalized answer lengths")
ax.set_xlabel("Words")
ax.set_ylabel("Count")
ax.legend()
plt.tight_layout()
plt.show()

compliance = {}
for split_name, summary in split_summaries.items():
    lengths = summary["target_lengths"]
    compliance[split_name] = sum(1 for x in lengths if x <= ANSWER_MAX_WORDS) / len(lengths)
compliance

## Evaluation helpers

Cell dưới đây dùng checkpoint sẵn có để đánh giá từng variant và gom metric thành bảng kết quả.
`B1` sẽ chạy trực tiếp trên model multimodal, `A1/A2` đọc checkpoint `.pth` từ thư mục `checkpoints/`.

In [ ]:
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

from src.data.medical_dataset import MedicalVQADataset
from src.engine.medical_eval import evaluate_multimodal_vqa, evaluate_vqa
from src.models.medical_vqa_model import MedicalVQAModelA
from src.models.multimodal_vqa import MultimodalVQA
from src.utils.visualization import MedicalImageTransform as MedicalTransform

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(cfg["model_a"]["phobert_model"])
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token or tokenizer.unk_token

transform = MedicalTransform(size=cfg["data"]["image_size"])
eval_dataset = MedicalVQADataset(
    hf_dataset=dataset["test"],
    tokenizer=tokenizer,
    transform=transform,
    max_seq_len=cfg["data"]["max_question_len"],
    max_ans_len=cfg["data"]["max_answer_len"],
    answer_max_words=ANSWER_MAX_WORDS,
)
eval_loader = DataLoader(eval_dataset, batch_size=cfg["train"].get("eval_batch_size", 8), collate_fn=lambda batch: {
    key: torch.stack([item[key] for item in batch]) if key in {"image", "input_ids", "attention_mask", "label_closed", "target_ids"} else [item[key] for item in batch]
    for key in batch[0]
})

def load_variant_a(variant_name):
    decoder_type = "lstm" if variant_name == "A1" else "transformer"
    model = MedicalVQAModelA(
        decoder_type=decoder_type,
        vocab_size=len(tokenizer),
        hidden_size=cfg["model_a"].get("hidden_size", 768),
        phobert_model=cfg["model_a"].get("phobert_model", "vinai/phobert-base"),
    ).to(DEVICE)
    ckpt_path = CHECKPOINT_DIR / f"medical_vqa_{variant_name}_best.pth"
    state_dict = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(state_dict)
    model.eval()
    return model

def evaluate_variant(variant_name):
    if variant_name in {"A1", "A2"}:
        model = load_variant_a(variant_name)
        return evaluate_vqa(
            model,
            eval_loader,
            DEVICE,
            tokenizer,
            beam_width=cfg["eval"].get("beam_width_a", 5),
            max_len=cfg["data"].get("max_answer_len", 20),
            max_words=ANSWER_MAX_WORDS,
        )

    wrapper = MultimodalVQA(model_id=cfg["model_b"]["model_name"])
    model, processor = wrapper.load_model()
    return evaluate_multimodal_vqa(
        model,
        eval_loader,
        DEVICE,
        processor,
        beam_width=cfg["eval"].get("beam_width_b", 1),
        max_words=ANSWER_MAX_WORDS,
    )

In [ ]:
variants = ["A1", "A2", "B1", "B2"]
results_by_variant = {}

for variant in variants:
    ckpt_required = variant in {"A1", "A2"}
    ckpt_path = CHECKPOINT_DIR / f"medical_vqa_{variant}_best.pth"
    if ckpt_required and not ckpt_path.exists():
        print(f"Skip {variant}: missing checkpoint {ckpt_path.name}")
        continue
    try:
        print(f"Evaluating {variant} ...")
        results_by_variant[variant] = evaluate_variant(variant)
    except Exception as exc:
        print(f"Variant {variant} failed: {exc}")

{k: {m: round(v[m], 4) for m in ['accuracy', 'f1', 'bleu4', 'rouge_l', 'bert_score', 'max_10_word_compliance_rate']} for k, v in results_by_variant.items()}

In [ ]:
metric_names = ["accuracy", "f1", "bleu4", "rouge_l", "bert_score"]
available_variants = list(results_by_variant.keys())

if available_variants:
    fig, ax = plt.subplots(figsize=(10, 5))
    x = range(len(available_variants))
    width = 0.15

    for idx, metric_name in enumerate(metric_names):
        values = [results_by_variant[v][metric_name] for v in available_variants]
        offsets = [pos + (idx - 2) * width for pos in x]
        ax.bar(offsets, values, width=width, label=metric_name)

    ax.set_xticks(list(x))
    ax.set_xticklabels(available_variants)
    ax.set_title("Metric comparison across variants")
    ax.set_ylabel("Score")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No evaluation results available yet.")

In [ ]:
if available_variants:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    closed_acc = [results_by_variant[v].get("closed", {}).get("accuracy", 0.0) for v in available_variants]
    open_bleu4 = [results_by_variant[v].get("open", {}).get("bleu4", 0.0) for v in available_variants]
    compliance = [results_by_variant[v].get("max_10_word_compliance_rate", 0.0) for v in available_variants]

    axes[0].bar(available_variants, closed_acc, color="#4C78A8")
    axes[0].set_title("Closed-question accuracy")
    axes[0].set_ylabel("Accuracy")

    axes[1].bar(available_variants, open_bleu4, color="#72B7B2")
    axes[1].set_title("Open-question BLEU-4")

    plt.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(available_variants, compliance, color="#54A24B")
    ax.set_ylim(0, 1.05)
    ax.set_title("Rate of outputs with <= 10 words")
    ax.set_ylabel("Compliance rate")
    plt.tight_layout()
    plt.show()

## Single-sample test cell

Cập nhật `TEST_VARIANT`, `TEST_IMAGE_PATH`, `TEST_QUESTION` rồi chạy cell dưới.
Với `A1/A2`, cần checkpoint tương ứng trong thư mục `checkpoints/`.

In [ ]:
from PIL import Image
import torchvision.transforms as T

TEST_VARIANT = "A2"
TEST_IMAGE_PATH = "data/sample.png"
TEST_QUESTION = "Hình ảnh này có bất thường không?"

def infer_single_sample(variant_name, image_path, question_vi):
    raw_image = Image.open(image_path).convert("RGB")
    image = raw_image.convert("L")
    tensor = transform(image).unsqueeze(0).to(DEVICE)
    encoded = tokenizer(
        question_vi,
        padding="max_length",
        truncation=True,
        max_length=cfg["data"]["max_question_len"],
        return_tensors="pt",
    )
    input_ids = encoded["input_ids"].to(DEVICE)
    attention_mask = encoded["attention_mask"].to(DEVICE)

    if variant_name in {"A1", "A2"}:
        model = load_variant_a(variant_name)
        logits_closed, pred_ids = model.inference(
            tensor,
            input_ids,
            attention_mask,
            beam_width=cfg["eval"].get("beam_width_a", 5),
            max_len=cfg["data"].get("max_answer_len", 20),
        )
        closed_label = torch.argmax(logits_closed, dim=-1).item()
        raw_output = tokenizer.decode(pred_ids[0], skip_special_tokens=True)
        final_answer = "có" if closed_label == 1 else "không"
        closed_starters = ("có ", "không ", "is ", "does ", "are ")
        if not question_vi.lower().startswith(closed_starters):
            final_answer = postprocess_answer(raw_output, max_words=ANSWER_MAX_WORDS)
        return raw_output, postprocess_answer(final_answer, max_words=ANSWER_MAX_WORDS)

    wrapper = MultimodalVQA(model_id=cfg["model_b"]["model_name"])
    model, processor = wrapper.load_model()
    prompt = wrapper.build_instruction_prompt(question_vi, language="vi", include_answer=False)
    inputs = processor(text=[prompt], images=[raw_image], return_tensors="pt", padding=True).to(DEVICE)
    output_ids = model.generate(**inputs, max_new_tokens=ANSWER_MAX_WORDS + 4, do_sample=False)
    raw_output = processor.batch_decode(output_ids[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]
    return raw_output, postprocess_answer(raw_output, max_words=ANSWER_MAX_WORDS)

raw_output, final_answer = infer_single_sample(TEST_VARIANT, TEST_IMAGE_PATH, TEST_QUESTION)
print("Question:", TEST_QUESTION)
print("Raw output:", raw_output)
print("Final answer:", final_answer)
print("Word count:", count_words(final_answer))
print("<= 10 words:", count_words(final_answer) <= ANSWER_MAX_WORDS)